Exercise

- Use message prefilling and stop sequences _only_ to get three different commands in a single response
- There shouldn't be any comments or explanation

In [7]:
# Create an API client and Helper Functions
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

token = os.environ["ANTHROPIC_AUTH_TOKEN"]
url = os.environ["ANTHROPIC_BASE_URL"]
model = os.environ["ANTHROPIC_MODEL"]

client = Anthropic(
    api_key=token,
    base_url=url,
    default_headers={"Authorization": f"Bearer {token}"}
)

def add_user_message(messages, content):
    user_message = { "role": "user", "content": content }
    messages.append(user_message)

def add_assistant_message(messages, content):
    assistant_message = { "role": "assistant", "content": content }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)
  return message.content[0].text

Solution

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```")

text = chat(messages, stop_sequences=["```"])
text.strip()

#The environment its not hitting Anthropic API directly, and mine does not support assistant message prefill

BadRequestError: Error code: 400 - {'error': {'message': '{"timestamp":"2026-08-14T23:38:35.636Z","path":"/ai-orchestration-api/v1/bedrock/invoke","error":"Client error with status 400 received from provider","history":[{"response":{"type":"error","error":{"type":"invalid_request_error","message":"This model does not support assistant message prefill. The conversation must end with a user message."},"request_id":"req_011Ce3XEQAnZNaQcmT9Cf6CV"},"status":400,"message":"Http Exception","name":"HttpException"}]}. Received Model Group=anthropic.claude-4-6-sonnet\nAvailable Model Group Fallbacks=None', 'type': 'None', 'param': 'None', 'code': '400'}}

In [5]:
from IPython.display import display, Markdown

Markdown(text)

Here are three short AWS CLI commands:

1. **List S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **List IAM users:**
```bash
aws iam list-users
```

Yes. Since assistant message prefill is blocked in your environment, you can get equivalent behavior by putting the formatting instruction in the user prompt instead of relying on prefill, and still use `stop_sequences` to cut off anything after the code block. It won't be *exactly* "prefill + stop sequences" as the exercise instructs, but it produces the same practical result: three short AWS CLI commands, no comments or explanation.

```python
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.

Respond with only the three commands wrapped in triple backticks (```), 
with no comments, explanation, or extra text before or after.
"""

add_user_message(messages, prompt)

text = chat(messages, stop_sequences=["```"])
text.strip()
```

This asks Claude to open the code block itself (since you can't prefill it), and the `stop_sequences=["```"]` still does its job: it stops generation the moment Claude hits the closing triple-backtick, so you don't get trailing commentary after the code block either.

A couple of notes on this approach:

Since there's no prefill, the very first `` ``` `` Claude generates is the *opening* fence, and the stop sequence will trigger on the *closing* fence, so `text` will include that leading `` ``` `` plus the three commands, and you'll want to strip it manually if you need clean output, e.g. `text.strip().lstrip("`").strip()` or a quick regex to pull out just the commands.

If you do want to genuinely satisfy the exercise's requirement of using prefill (not just stop sequences), that's only possible if you can get prefill working at all, which means either switching `model` to a real Anthropic API model string as I mentioned, or using a different key/environment that talks to the Anthropic API directly rather than through that Bedrock proxy. If you want, I can help you verify whether changing `model` fixes the prefill error, since that would let you complete the exercise exactly as written.